# Lecture 4: Reinforcement Learning
**Prof. Dr. Björn Sprungk — Methods of Machine Learning, Summer Term 2025**

---

## Contents
1. [Introduction](#introduction)
2. [4.1 Markov Decision Processes (MDPs)](#mdp)
3. [4.2 Dynamic Programming](#dp)
4. [4.3 Temporal-Difference Learning](#td)

## 1. Introduction <a id='introduction'></a>

**Reinforcement Learning (RL)** is a learning paradigm where an **agent** learns by interacting with an **environment**.

### Examples
- Learning to play chess
- Self-control of robots
- A young gazelle learning to walk within its first hour

### Core Task
> By interacting with its environment, an **agent** is supposed to learn actions/strategies to **maximize its reward**.

The agent-environment loop:
```
Agent --[action aₜ]--> Environment
Agent <--[reward Rₜ, state Sₜ]-- Environment
```

## 2. Markov Decision Processes (MDPs) <a id='mdp'></a>

### 2.1 Setting

An MDP is defined by three finite sets (for simplicity):
- **S** — set of possible states $s \in \mathcal{S}$
- **A** — set of possible actions $a \in \mathcal{A}$ (may depend on state: $\mathcal{A}(s)$)
- **R** — set of possible rewards $r \in \mathcal{R}$

### 2.2 Agent-Environment Interface

The agent interacts in discrete time $t \in \{0, 1, 2, 3, \ldots\}$:
1. At $t=0$: agent observes state $s_0$, chooses action $a_0$
2. Environment returns new state $s_1$ and reward $r_1$
3. Repeat: from state $s_t$ and reward $r_t$, choose $a_t$ → get $(s_{t+1}, r_{t+1})$

This produces a **trajectory**:
$$s_0, a_0, r_1, s_1, a_1, r_2, s_2, a_2, r_3, \ldots$$

Treated as a **stochastic process**: $S_0, A_0, R_1, S_1, A_1, R_2, \ldots$ (all random variables).

### 2.3 The Markov Property

$$\mathbb{P}(S_{t+1}=s, R_{t+1}=r \mid A_t, S_t, R_t, \ldots, S_0) = \mathbb{P}(S_{t+1}=s, R_{t+1}=r \mid A_t, S_t)$$

> **Meaning:** The next state and reward depend **only on the current state and action** — not the full history.

### 2.4 Transition Probabilities

The dynamics of an MDP are fully described by:
$$p(s', r \mid s, a) := \mathbb{P}(S_{t+1}=s', R_{t+1}=r \mid A_t=a, S_t=s)$$

This is a **stochastic kernel**: $\sum_{s', r} p(s', r \mid s, a) = 1$.

From this we can derive:
- **Marginal state-transition probability:** $p(s' \mid s,a) = \sum_{r} p(s',r\mid s,a)$
- **Expected reward (given next state):** $r(s,a,s') = \sum_r r \frac{p(s',r\mid s,a)}{p(s'\mid s,a)}$
- **Expected reward:** $r(s,a) = \sum_{s',r} r\, p(s',r\mid s,a)$

### 2.5 Example: Recycling Robot

A robot collects soda cans, powered by a rechargeable battery.

| Component | Description |
|-----------|-------------|
| **States** | $\mathcal{S} = \{\text{high}, \text{low}\}$ (battery level) |
| **Actions** | `search`, `wait`, `recharge` (only `recharge` available from `low`) |
| **Rewards** | Positive for collecting cans; $-3$ if battery depleted while searching |

**Transition probabilities:**
- Searching from `high`: stays `high` w.p. $\alpha$, drops to `low` w.p. $1-\alpha$
- Searching from `low`: stays `low` w.p. $\beta$, depletes & recharges to `high` w.p. $1-\beta$ (reward $= -3$)
- Waiting: stays in same state w.p. 1, reward $r_{\text{wait}}$
- Recharging from `low`: always goes to `high`, reward $= 0$

In [ ]:
import numpy as np

# Recycling Robot MDP parameters
alpha = 0.7   # P(stay high | high, search)
beta  = 0.5   # P(stay low  | low,  search)
r_search = 3  # expected cans when searching
r_wait   = 1  # expected cans when waiting

# States: 0=high, 1=low
# Actions per state:
#   high: 0=search, 1=wait
#   low:  0=search, 1=wait, 2=recharge

# Transition table: p[s][a] = [(prob, next_state, reward), ...]
transitions = {
    'high': {
        'search':   [(alpha,   'high', r_search),
                     (1-alpha, 'low',  r_search)],
        'wait':     [(1.0,     'high', r_wait)],
    },
    'low': {
        'search':   [(beta,    'low',  r_search),
                     (1-beta,  'high', -3)],
        'wait':     [(1.0,     'low',  r_wait)],
        'recharge': [(1.0,     'high', 0)],
    }
}

print("Recycling Robot MDP transitions:")
for state, actions in transitions.items():
    for action, outcomes in actions.items():
        for prob, next_s, reward in outcomes:
            print(f"  ({state}, {action}) -> {next_s}: p={prob}, r={reward}")

### 2.6 Policies

A **policy** $\pi$ defines the agent's behaviour. We use **stochastic policies**:
$$\pi(a \mid s) = \mathbb{P}(A_t = a \mid S_t = s) \qquad \forall a \in \mathcal{A}, s \in \mathcal{S}$$

This is again a stochastic kernel: $\sum_a \pi(a \mid s) = 1$.

Expected reward under policy $\pi$ in state $s$:
$$\mathbb{E}_\pi[R_{t+1} \mid S_t = s] = \sum_{a,s',r} r\, p(s',r \mid s,a)\, \pi(a \mid s)$$

### 2.7 Returns and Discounting

The **return** $G_t$ is the cumulative future reward. For **continuous tasks** (no terminal state), we use **discounted returns**:

$$G_t = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + \cdots = \sum_{k=0}^{\infty} \gamma^k R_{t+k+1}$$

where $\gamma \in [0,1]$ is the **discount rate**.

| $\gamma$ | Interpretation |
|----------|----------------|
| $\gamma = 0$ | Myopic: only immediate reward matters |
| $\gamma \to 1$ | Far-sighted: future rewards matter equally |

**Key recursive relation:**
$$G_t = R_{t+1} + \gamma G_{t+1}$$

### 2.8 Value Functions

The **state-value function** under policy $\pi$:
$$v_\pi(s) = \mathbb{E}_\pi[G_t \mid S_t = s]$$

The **action-value function** under policy $\pi$:
$$q_\pi(s, a) = \mathbb{E}_\pi[G_t \mid S_t = s, A_t = a]$$

> $v_\pi(s)$ quantifies **how good it is** for the agent to be in state $s$ under policy $\pi$.

### 2.9 Bellman Equation

The value function satisfies the **Bellman equation** for all $s \in \mathcal{S}$:

$$v_\pi(s) = \sum_{a \in \mathcal{A}} \sum_{s', r} \bigl(r + \gamma\, v_\pi(s')\bigr)\, p(s',r \mid s,a)\, \pi(a\mid s)$$

This is a **linear system** $(I - \gamma P)v_\pi = b$ and $v_\pi$ is its **unique solution**.

### 2.10 Optimal Value Functions

Policy ordering: $\pi \geq \pi'$ iff $v_\pi(s) \geq v_{\pi'}(s)$ for all $s$.

$$v^*(s) = \max_\pi v_\pi(s), \qquad q^*(s,a) = \max_\pi q_\pi(s,a)$$

Relations:
$$v^*(s) = \max_a q^*(s,a)$$
$$q^*(s,a) = r(s,a) + \gamma \sum_{s'} p(s'\mid s,a)\, v^*(s')$$

**Bellman Optimality Equation** (unique solution for $v^*$):
$$v^*(s) = \max_{a \in \mathcal{A}} \left[ r(s,a) + \gamma \sum_{s'} p(s'\mid s,a)\, v^*(s') \right]$$

The **optimal (greedy) policy**:
$$\pi^*(a\mid s) = \begin{cases} 1 & a = \arg\max_{a'} q^*(s,a') \\ 0 & \text{else} \end{cases}$$

## 3. Dynamic Programming <a id='dp'></a>

**Dynamic Programming (DP)** algorithms compute optimal policies given a **perfect model** of the environment (i.e., $p$ and $r$ are fully known).

### 3.1 Value Iteration

Based on a **contractive fixed-point iteration**. Define the operator $T: \mathbb{R}^\mathcal{S} \to \mathbb{R}^\mathcal{S}$:
$$Tv(s) := \max_{a \in \mathcal{A}} \left[ r(s,a) + \gamma \sum_{s'} p(s'\mid s,a)\, v(s') \right]$$

**Contractivity:** $\|Tv - Tv'\|_\infty \leq \gamma \|v - v'\|_\infty$

By the **Banach Fixed Point Theorem**, $v^*$ is the unique fixed point and $\|T^n v - v^*\|_\infty \leq \gamma^n \|v - v^*\|_\infty$.

**Algorithm:**
```
Initialize: v₀(s) = 0 for all s, choose ε > 0, δ = ε(1-γ)/γ
While δ > ε(1-γ)/(2γ):
    v_{k+1}(s) = max_a [ r(s,a) + γ Σ_{s'} p(s'|s,a) v_k(s') ]  for all s
    δ = max_s |v_{k+1}(s) - v_k(s)|
    k += 1
```
Final iterate $v_K$ satisfies $\|v_K - v^*\|_\infty \leq \varepsilon/2$.

**Extracting the policy:**
```
q(s,a) = r(s,a) + Σ_{s'} p(s'|s,a) v_K(s)
π(a|s) = 1 if a = argmax q(s,a), else 0
```
This gives $\|v_\pi - v^*\|_\infty \leq \varepsilon$.

In [ ]:
import numpy as np

def value_iteration(states, actions_fn, reward_fn, transition_fn, gamma=0.9, eps=1e-6):
    """
    Value iteration algorithm.
    
    Args:
        states:        list of states
        actions_fn:    function s -> list of actions available in s
        reward_fn:     function (s, a) -> expected reward r(s, a)
        transition_fn: function (s, a) -> list of (prob, next_state)
        gamma:         discount factor
        eps:           convergence threshold
    Returns:
        v:  optimal value function (dict)
        pi: greedy optimal policy (dict: s -> best action)
    """
    v = {s: 0.0 for s in states}
    threshold = eps * (1 - gamma) / (2 * gamma)

    while True:
        delta = 0
        v_new = {}
        for s in states:
            q_values = []
            for a in actions_fn(s):
                q = reward_fn(s, a) + gamma * sum(
                    p * v[s_next] for p, s_next in transition_fn(s, a)
                )
                q_values.append((q, a))
            v_new[s] = max(q for q, _ in q_values)
            delta = max(delta, abs(v_new[s] - v[s]))
        v = v_new
        if delta <= threshold:
            break

    # Extract greedy policy
    pi = {}
    for s in states:
        best_a = max(actions_fn(s),
                     key=lambda a: reward_fn(s, a) + gamma * sum(
                         p * v[s_next] for p, s_next in transition_fn(s, a)))
        pi[s] = best_a

    return v, pi


# --- Recycling Robot example ---
alpha, beta = 0.7, 0.5
r_search, r_wait = 3.0, 1.0

states = ['high', 'low']
actions = {'high': ['search', 'wait'], 'low': ['search', 'wait', 'recharge']}

def reward_fn(s, a):
    if a == 'search':
        if s == 'high': return r_search
        else:           return beta * r_search + (1 - beta) * (-3)
    elif a == 'wait':    return r_wait
    else:                return 0.0

def transition_fn(s, a):
    if a == 'search':
        if s == 'high': return [(alpha, 'high'), (1-alpha, 'low')]
        else:           return [(beta, 'low'), (1-beta, 'high')]
    elif a == 'wait':    return [(1.0, s)]
    else:                return [(1.0, 'high')]  # recharge

v, pi = value_iteration(states, lambda s: actions[s], reward_fn, transition_fn, gamma=0.9)

print("Optimal value function:")
for s, val in v.items():
    print(f"  v*({s}) = {val:.4f}")
print("\nOptimal policy:")
for s, a in pi.items():
    print(f"  π*({s}) = {a}")

## 4. Temporal-Difference Learning <a id='td'></a>

**DP requires full knowledge** of $p$ and $r$. TD learning removes this requirement by using **sample-based approximations**.

Recall the Bellman equations in expectation form:
$$v(s) = \mathbb{E}_\pi[R_{t+1} + \gamma v(S_{t+1}) \mid S_t = s]$$
$$q^*(s,a) = \mathbb{E}\left[R_{t+1} + \gamma \max_{a'} q^*(S_{t+1}, a') \mid S_t=s, A_t=a\right]$$

### 4.1 Robbins-Monro Algorithm

To find the root of $G(x) = \mathbb{E}[g(x, Z)] = 0$ using i.i.d. samples $z_n \sim \nu$:
$$x_n = x_{n-1} - \alpha_n\, g(x_{n-1}, z_n)$$

Under suitable conditions: $x_n \xrightarrow{a.s.} x^*$.

> **Note:** SGD is just a special case of Robbins-Monro (finding the root of $\nabla L = 0$).

### 4.2 TD(0) — Policy Evaluation

Estimates $v_\pi$ without knowing $p$ or $r$.

**Update rule** (at each step $n$):
$$v_{n+1}(s_n) = v_n(s_n) + \alpha_{n+1} \underbrace{\bigl(r_{n+1} + \gamma v_n(s_{n+1}) - v_n(s_n)\bigr)}_{\text{TD error}}$$

**Convergence:** Given $\sum_n \alpha_n = \infty$ and $\sum_n \alpha_n^2 < \infty$, then $v_n(s) \xrightarrow{a.s.} v_\pi(s)$.

### 4.3 Q-Learning — Off-Policy TD Control

Learns $q^*$ directly, regardless of the behaviour policy.

**Update rule:**
$$q_{n+1}(s_n, a_n) = q_n(s_n, a_n) + \alpha_{n+1}\left(r_{n+1} + \gamma \max_{a'} q_n(s_{n+1}, a') - q_n(s_n, a_n)\right)$$

Policy is updated greedily after each step: $\pi_{n+1}(\cdot\mid s) = \arg\max_{a'} q_{n+1}(s, a')$.

Under mild conditions: $q_n(s,a) \xrightarrow{a.s.} q^*(s,a)$.

### 4.4 SARSA — On-Policy TD Control

A simplification of Q-learning when using the greedy policy:
$$q_{n+1}(s_n, a_n) = q_n(s_n, a_n) + \alpha_{n+1}\bigl(r_{n+1} + q_n(s_{n+1}, a_{n+1}) - q_n(s_n, a_n)\bigr)$$

where $a_{n+1} \sim \pi_{n+1}(\cdot \mid s_{n+1})$. Name comes from the tuple used: **(S, A, R, S', A')**.

### 4.5 Summary Comparison

| Algorithm | Knows $p, r$? | Estimates | On/Off-policy |
|-----------|--------------|-----------|---------------|
| Value Iteration | ✅ Yes | $v^*$ | — |
| TD(0) | ❌ No | $v_\pi$ | On-policy |
| Q-Learning | ❌ No | $q^*$ | Off-policy |
| SARSA | ❌ No | $q^*$ | On-policy |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def q_learning(env_step, states, actions, gamma=0.9, n_episodes=2000, alpha_fn=None):
    """
    Q-Learning algorithm.
    
    Args:
        env_step: function (s, a) -> (next_state, reward)
        states, actions: lists
        gamma: discount factor
        n_episodes: number of update steps
        alpha_fn: function n -> step size (default: 1/n)
    Returns:
        q: action-value table (dict)
        pi: greedy policy (dict)
    """
    if alpha_fn is None:
        alpha_fn = lambda n: 1.0 / n

    q = {(s, a): 0.0 for s in states for a in actions}
    counts = {(s, a): 0 for s in states for a in actions}

    s = states[0]  # start state
    for n in range(1, n_episodes + 1):
        # Epsilon-greedy action selection
        eps = max(0.1, 1.0 / (n ** 0.5))
        if np.random.rand() < eps:
            a = np.random.choice(actions)
        else:
            a = max(actions, key=lambda a_: q[(s, a_)])

        s_next, r = env_step(s, a)
        counts[(s, a)] += 1
        alpha = alpha_fn(counts[(s, a)])

        best_next = max(q[(s_next, a_)] for a_ in actions)
        q[(s, a)] += alpha * (r + gamma * best_next - q[(s, a)])

        s = s_next

    pi = {s: max(actions, key=lambda a: q[(s, a)]) for s in states}
    return q, pi


# Recycling Robot: stochastic simulation
def robot_env_step(s, a):
    if a == 'search':
        if s == 'high':
            s_next = 'high' if np.random.rand() < alpha else 'low'
            return s_next, r_search
        else:
            if np.random.rand() < beta:
                return 'low', r_search
            else:
                return 'high', -3
    elif a == 'wait':
        return s, r_wait
    else:  # recharge
        return 'high', 0

all_actions = ['search', 'wait', 'recharge']
q_est, pi_est = q_learning(robot_env_step, states, all_actions, gamma=0.9, n_episodes=10000)

print("Q-Learning results (Recycling Robot):")
print("\nAction-value estimates q(s,a):")
for s in states:
    for a in all_actions:
        print(f"  q({s}, {a:8s}) = {q_est[(s,a)]:.4f}")
print("\nLearned policy:")
for s, a in pi_est.items():
    print(f"  π*({s}) = {a}")

## 5. Beyond Tabular Methods

Tabular RL stores $\pi$ and $q_\pi$ as matrices of size $|\mathcal{A}| \times |\mathcal{S}|$.

**Problem:** For large/continuous state spaces this is infeasible.

> Example: The game of Go has $|\mathcal{S}| \approx 10^{170}$ states — a table is impossible.

**Solution:** Use **deep neural networks** to approximate $\pi$ and $q$.
- E.g., Google DeepMind's **Deep Q-Networks (DQN)** — trained using Q-learning on the neural network weights.
- The network parameters are optimized jointly with the RL learning process.